In [2]:
# CELL 1 - Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# CELL 2 - Load the large dataset
df = pd.read_csv('Heart_Disease_and_Hospitals.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumns:")
print(df.columns.tolist())

print("\nHeart disease distribution:")
print(df['heart_disease'].value_counts())

# CELL 3 - Feature Engineering & Cleaning
# Keep only numerical + meaningful categorical features
feature_cols = [
    'age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level', 'gender'
]

# Encode gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0, 'Other': 1})  # safe mapping

# Extract features and target
X = df[feature_cols]
y = df['heart_disease']

# Handle any missing values (just in case)
X = X.fillna(X.median())

print(f"\nUsing features: {feature_cols}")
print(f"Final X shape: {X.shape}")

# CELL 4 - Take 500 stratified balanced samples
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=500, random_state=42, stratify=y
)

print(f"\nSelected 500 balanced samples")
print("Class distribution in sample:")
print(pd.Series(y_sample).value_counts())

# CELL 5 - Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

# Scale features (needed for LR and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# CELL 6 - 1. Logistic Regression
print("\n" + "="*60)
print("1. LOGISTIC REGRESSION")
print("="*60)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test_scaled))
print(f"Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)")

# CELL 7 - 2. Random Forest (King of tabular data)
print("\n" + "="*60)
print("2. RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest Accuracy:       {acc_rf:.4f} ({acc_rf*100:.1f}%)")

# Feature importance
print("\nFeature Importance (Random Forest):")
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.round(4))

# CELL 8 - 3. Support Vector Machine
print("\n" + "="*60)
print("3. SUPPORT VECTOR MACHINE (RBF)")
print("="*60)

svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train_scaled, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test_scaled))
print(f"SVM Accuracy:                 {acc_svm:.4f} ({acc_svm*100:.1f}%)")

# CELL 9 - Final Results
print("\n" + "="*70)
print("FINAL RESULTS - 500 SAMPLES FROM 10,000-RECORD DATASET")
print("="*70)
print(f"Logistic Regression   → {acc_lr:.4f}")
print(f"Random Forest         → {acc_rf:.4f}  ← Almost certainly the winner")
print(f"SVM (RBF)             → {acc_svm:.4f}")
print("="*70)

best = max(acc_lr, acc_rf, acc_svm)
if best == acc_rf:
    print("RANDOM FOREST IS THE BEST MODEL (as expected on real-world data)!")
elif best == acc_svm:
    print("SVM wins!")
else:
    print("Logistic Regression wins!")

# Bonus: Predict on a real patient
print(f"\nExample from test set:")
sample = X_test.iloc[0:1]
pred = rf.predict(sample)[0]
true = y_test.iloc[0]
age = int(sample['age'].values[0])
print(f"Patient age {age}, True label: {true} → RF predicts: {pred} → {'CORRECT' if pred==true else 'WRONG'}")

Dataset shape: (10000, 15)

First 5 rows:
        full_name  age country          state  blood_pressure  cholesterol  \
0   Alice Johnson   21     USA           Utah      138.068129   270.564349   
1    Bob Williams   44     USA   North Dakota      100.383639   160.910030   
2    Oliver Jones   22     USA           Iowa      145.618318   208.490251   
3     Quinn Davis   19     USA      Louisiana      152.577870   348.153738   
4  Alice Martinez   64     USA  Massachusetts      111.793570   200.628925   

         bmi  glucose_level  gender first_name last_name  \
0  36.101427      63.487954  Female      Alice   Johnson   
1  14.330471     187.008977  Female        Bob  Williams   
2  20.277098      35.903576  Female     Oliver     Jones   
3  62.712693      31.495166  Female      Quinn     Davis   
4  14.468485     154.513129  Female      Alice  Martinez   

                       hospital       treatment  \
0            Starlight Hospital   Physiotherapy   
1           Maple Leaf Hos